# Miso Agent - Ollama Demo

使用 **Ollama** 本地模型运行 miso Agent，无需 API Key。

**前置条件：**
- 已安装并运行 [Ollama](https://ollama.com)（默认 `http://localhost:11434`）
- 已拉取支持 tool calling 的模型，例如：`ollama pull qwen2.5:7b`

## 1. 导入依赖

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

from miso import agent, tool

## 2. 配置 Ollama Agent

Ollama 运行在本地，无需 API Key，只需指定模型名称。
推荐支持 tool calling 的模型：`qwen2.5:7b`、`llama3.1:8b`、`mistral` 等。

In [2]:
my_agent = agent()

my_agent.provider = "ollama"           # 使用 Ollama 本地推理
my_agent.model = "qwen3:8b"         # ⚠️ 必须用支持 tool calling 的模型
my_agent.max_iterations = 6           #    支持: qwen2.5, llama3.1, mistral, command-r 等
                                       #    不支持: deepseek-r1, phi3, gemma 等

print(f"Provider: {my_agent.provider}")
print(f"Model:    {my_agent.model}")

Provider: ollama
Model:    qwen3:8b


## 3. 定义工具 (Tools)

和 OpenAI demo 一样，定义 `get_weather` 和 `calculate` 两个工具。

In [3]:
# ---------- 工具 1: 查天气 ----------
def get_weather(city: str) -> dict:
    """Get the current weather for a given city."""
    fake_data = {
        "tokyo": {"temp": "22°C", "condition": "Sunny"},
        "beijing": {"temp": "18°C", "condition": "Cloudy"},
        "new york": {"temp": "15°C", "condition": "Rainy"},
    }
    info = fake_data.get(city.lower(), {"temp": "20°C", "condition": "Unknown"})
    return {"city": city, "temperature": info["temp"], "condition": info["condition"]}


# ---------- 工具 2: 计算器 ----------
def calculate(expression: str) -> dict:
    """Evaluate a mathematical expression and return the result."""
    try:
        result = eval(expression, {"__builtins__": {}})
        return {"expression": expression, "result": str(result)}
    except Exception as e:
        return {"expression": expression, "error": str(e)}


# 注册工具
my_agent.toolkit.register(
    tool.from_callable(get_weather, name="get_weather",
                       description="Get the current weather for a given city.")
)
my_agent.toolkit.register(
    tool.from_callable(calculate, name="calculate",
                       description="Evaluate a mathematical expression (e.g. '123 * 456').")
)

print("已注册工具:", list(my_agent.toolkit.tools.keys()))

已注册工具: ['get_weather', 'calculate']


## 4. 运行 Agent

Ollama 的 tool calling 流程和 OpenAI 完全一致，miso 内部自动处理协议差异。

In [4]:
# callback: 打印 Agent 运行过程
def print_events(event):
    t = event["type"]
    if t == "run_started":
        print(f"🚀 Agent 启动  (model={event.get('model')})")
    elif t == "tool_call":
        print(f"🔧 调用工具: {event['tool_name']}({event.get('arguments', '')})")
    elif t == "tool_result":
        print(f"   ↳ 工具返回: {event.get('result', '')}")
    elif t == "observation":
        print(f"   👁 观察: {event.get('content', '')}")
    elif t == "final_message":
        print(f"\n✅ 最终回复:\n{event.get('content', '')}")
    elif t == "run_completed":
        print(f"\n--- Agent 完成 (共 {event['iteration'] + 1} 轮) ---")

In [5]:
# 示例 1: 查询天气
conversation = my_agent.run(
    messages=[{"role": "user", "content": "What is the weather in Tokyo?"}],
    callback=print_events,
)

🚀 Agent 启动  (model=qwen3:8b)
🔧 调用工具: get_weather({'city': 'Tokyo'})
   ↳ 工具返回: {'city': 'Tokyo', 'temperature': '22°C', 'condition': 'Sunny'}

✅ 最终回复:
Okay, the user asked for the weather in Tokyo. I called the get_weather function with Tokyo as the city parameter. The response came back with a temperature of 22°C and sunny conditions. Now I need to present this information in a clear and friendly way. Let me check if all the details are there. Yep, city, temperature, and condition are included. I'll format it like a typical weather report, mentioning the city, temperature, and the condition. Make sure to use proper units and a positive tone since it's sunny. Alright, that should cover the user's query effectively.
The current weather in Tokyo is **22°C** with **Sunny** conditions. ☀️

--- Agent 完成 (共 2 轮) ---


In [6]:
conversation

([{'role': 'user', 'content': 'What is the weather in Tokyo?'},
  {'role': 'assistant',
   'content': '',
   'tool_calls': [{'id': 'call_qjqe33dr',
     'function': {'index': 0,
      'name': 'get_weather',
      'arguments': {'city': 'Tokyo'}}}]},
  {'role': 'tool',
   'tool_call_id': 'call_qjqe33dr',
   'content': '{"city": "Tokyo", "temperature": "22°C", "condition": "Sunny"}'},
  {'role': 'assistant',
   'content': "Okay, the user asked for the weather in Tokyo. I called the get_weather function with Tokyo as the city parameter. The response came back with a temperature of 22°C and sunny conditions. Now I need to present this information in a clear and friendly way. Let me check if all the details are there. Yep, city, temperature, and condition are included. I'll format it like a typical weather report, mentioning the city, temperature, and the condition. Make sure to use proper units and a positive tone since it's sunny. Alright, that should cover the user's query effectively.\nT

In [21]:
# 示例 2: 数学计算
conversation2 = my_agent.run(
    messages=[{"role": "user", "content": "帮我计算 123 * 456 + 789"}],
    callback=print_events,
)

🚀 Agent 启动  (model=qwen3:8b)
🔧 调用工具: calculate({'expression': '123 * 456 + 789'})
   ↳ 工具返回: {'expression': '123 * 456 + 789', 'result': '56877'}

✅ 最终回复:
好的，用户让我计算123乘以456再加上789的结果。首先，我需要确认用户的需求是否正确理解。用户给出的表达式是123 * 456 + 789，可能需要按照运算顺序先乘后加，所以先算乘法，再算加法。

接下来，我需要调用calculate函数来处理这个数学表达式。确保传递给函数的参数是正确的字符串形式，即"123 * 456 + 789"。然后，函数会返回结果，比如这里给出的示例结果是56877。我需要验证这个结果是否正确，可能手动计算或者用其他方法确认。例如，123乘以456等于56088，再加上789确实得到56877。最后，将结果以自然的方式回复给用户，确保清晰易懂。
123乘以456再加上789的结果是56877。计算过程遵循数学运算顺序，先进行乘法运算（123×456=56088），再执行加法运算（56088+789=56877）。

--- Agent 完成 (共 2 轮) ---


## 5. 多轮对话

In [20]:
# 在天气对话基础上继续追问
conversation.append({"role": "user", "content": "那 Beijing 呢？和 Tokyo 比哪个更热？"})

conversation = my_agent.run(
    messages=conversation,
    callback=print_events,
)

🚀 Agent 启动  (model=qwen3:8b)
🔧 调用工具: get_weather({'city': 'Beijing'})
   ↳ 工具返回: {'city': 'Beijing', 'temperature': '18°C', 'condition': 'Cloudy'}

✅ 最终回复:
Okay, the user first asked about the weather in Tokyo, and I provided the details: 22°C and sunny. Then they asked about Beijing and which city is hotter. I called the get_weather function for Beijing and got 18°C with cloudy conditions.

Now, I need to compare the two temperatures. Tokyo is 22°C, Beijing is 18°C. So Tokyo is warmer. The user might be planning to travel or dress appropriately, so mentioning the temperature difference makes sense. I should state the current weather for both cities and then clearly state which is hotter. Keep it straightforward and helpful.
The current weather in Beijing is **18°C** with **Cloudy** conditions. ☁️

Comparing the two: **Tokyo (22°C)** is warmer than **Beijing (18°C)**. 🌡️

--- Agent 完成 (共 2 轮) ---
